# ImageBind for Multimodal Highlight Extraction
## Complete Setup and Implementation Guide

**Project:** Intelligent Multimodal Approach for Automatic Generation of High-Impact Shorts

**Architecture:** ImageBind (One Embedding Space for Vision + Audio + Text)

**Advantages:**
-  Unified embedding space (vision, audio, text aligned)
-  Native audio understanding (critical for highlights)
-  Zero-shot cross-modal retrieval
-  Clean research framework
-  Lighter than VideoLLaMA2

---

## Part 1: Environment Setup

In [1]:
# Check system
import sys
import torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(" No GPU")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.9.0+cu128
CUDA Available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Part 2: Install Dependencies

In [2]:
%pip install whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for whisper: filename=whisper-1.1.10-py3-none-any.whl size=41120 sha256=88cedebe9329a16fe7f99a73c4b694d1398fc698c475555755d507f72d350c17
  Stored in directory: /root/.cache/pip/wheels/34/b8/4e/9c4c3351d670e06746a340fb4b7d854c76517eec225e5b32b1
Successfully built whisper


In [3]:
# Install PyTorch (if not already installed)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install pytorchvideo ftfy timm einops fvcore decord
%pip install torchaudio  # Ensure torchaudio is present for audio processing

print("Installing ImageBind dependencies...")

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 122.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 21.4 MB/s eta 0:00:00:00:0100:01
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=f74d2fa2b94ff85bdaf42a40ffeb33eae25f3986b8310fa8181ff17e3ab7b713
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.

In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_path = "/content/drive/MyDrive/PFA/ImageBind"
os.makedirs(project_path, exist_ok=True)

%cd {project_path}

%pwd

Mounted at /content/drive
/content/drive/MyDrive/PFA/ImageBind


'/content/drive/MyDrive/PFA/ImageBind'

In [ ]:
# Install ImageBind
!git clone https://github.com/facebookresearch/ImageBind.git
%cd ImageBind
!pip install -e .

Cloning into 'ImageBind'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 187 (delta 84), reused 54 (delta 53), pack-reused 67 (from 3)
Receiving objects: 100% (187/187), 2.65 MiB | 19.26 MiB/s, done.
Resolving deltas: 100% (92/92), done.
/content/drive/MyDrive/PFA/ImageBind/ImageBind
Obtaining file:///content/drive/MyDrive/PFA/ImageBind/ImageBind
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/pytorchvideo.git (to revision 6cdc929315aab1b5674b6dcf73b16ec99147735f) to /tmp/pip-install-35wmm1b1/pytorchvideo_046632b2c39b4e7490dd05964ba817dc
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorchvideo.git /tmp/pip-install-35wmm1b1/pytorchvideo_046632b2c39b4e7490dd05964ba817dc
  Running command git rev-parse -q --verify 'sha^6cdc929315aab1b5674b6dcf73b16ec99147735f'
  Running command git fetch -q https://g

In [6]:
# Install additional dependencies for video/audio processing
!pip install -q decord openai-whisper librosa soundfile opencv-python numpy scipy matplotlib tqdm

print("✓ All dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 42.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✓ All dependencies installed


In [7]:
# 1. Clean up existing broken attempts
!pip uninstall -y pytorchvideo numpy torch 

# 2. Install modern versions directly
!pip install torch>=2.0.0 torchvision torchaudio
!pip install git+https://github.com/facebookresearch/pytorchvideo.git@6cdc929315aab1b5674b6dcf73b16ec99147735f
!pip install timm ftfy regex einops iopath
!pip install "numpy>=1.19" types-regex

Found existing installation: pytorchvideo 0.1.5
Uninstalling pytorchvideo-0.1.5:
  Successfully uninstalled pytorchvideo-0.1.5
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: torch 2.9.0+cu128
Uninstalling torch-2.9.0+cu128:
  Successfully uninstalled torch-2.9.0+cu128
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
  Cloning https://github.com/facebookresearch/pytorchvideo.git (to revision 6cdc929315aab1b5674b6dcf73b16ec99147735f) to /tmp/pip-req-build-cjh4l89n
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorchvideo.git /tmp/pip-req-build-cjh4l89

In [8]:
# Fix NumPy conflict
%pip uninstall numpy -y
%pip install "numpy>=1.26,<2.1"


# IMPORTANT: Restart kernel after this!

Found existing installation: numpy 2.4.2
Uninstalling numpy-2.4.2:
  Successfully uninstalled numpy-2.4.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 54.1 MB/s eta 0:00:0000:0100:01


In [1]:
# Verify
import numpy as np
print(f"NumPy: {np.__version__}") 

NumPy: 2.0.2


In [2]:
%pip install --force-reinstall numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.6 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26

In [1]:
# Verify
import numpy as np
print(f"NumPy: {np.__version__}") 

NumPy: 1.26.4


In [2]:
# Verify
import torch as np
print(f"PyTorch: {np.__version__}") 

PyTorch: 2.9.0+cu128


In [3]:
!pip uninstall timm -y
!pip install timm==0.9.16

Found existing installation: timm 1.0.24
Uninstalling timm-1.0.24:
  Successfully uninstalled timm-1.0.24
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 97.4 MB/s eta 0:00:00


## Part 3: Load ImageBind Model

In [2]:
# SETUP FOR DRIVE - Run this cell

import os
import sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your folder
pfa_path = "/content/drive/MyDrive/PFA/ImageBind"
imagebind_path = os.path.join(pfa_path, "ImageBind")

# Create PFA folder if doesn't exist
os.makedirs(pfa_path, exist_ok=True)
os.chdir(pfa_path)

# Clone ImageBind if not already there
if not os.path.exists(imagebind_path):
    print("Cloning ImageBind to Drive...")
    !git clone https://github.com/facebookresearch/ImageBind.git
    print("✓ Cloned")
else:
    print("✓ ImageBind already exists in Drive")

# Navigate to ImageBind
os.chdir(imagebind_path)

# Install
print("\nInstalling ImageBind...")
!pip install -e .

# Add to Python path
sys.path.insert(0, imagebind_path)

print(f"\n✓ Working directory: {os.getcwd()}")
print(f"✓ ImageBind path added to sys.path")

# Test import
try:
    from imagebind import data
    print("✓ Import successful!")
except Exception as e:
    print(f" Import failed: {e}")
    print("Restart kernel, then run next cell")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ ImageBind already exists in Drive

Installing ImageBind...
Obtaining file:///content/drive/MyDrive/PFA/ImageBind/ImageBind
  Preparing metadata (setup.py) ... done
  Using cached pytorchvideo-0.1.5-py3-none-any.whl
  Running setup.py develop for imagebind

✓ Working directory: /content/drive/MyDrive/PFA/ImageBind/ImageBind
✓ ImageBind path added to sys.path
✓ Import successful!


In [3]:
import torch
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Loading ImageBind model...")
print("This downloads ~1.2GB and takes 2-3 minutes...\n")

# Load model
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(device)

print("✓ ImageBind model loaded successfully!")
print(f"Device: {device}")

# Check memory
if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated(0) / 1e9
    print(f"GPU Memory Used: {mem:.2f} GB")

Loading ImageBind model...
This downloads ~1.2GB and takes 2-3 minutes...



100%|██████████| 4.47G/4.47G [01:06<00:00, 72.2MB/s]


✓ ImageBind model loaded successfully!
Device: cuda:0
GPU Memory Used: 4.88 GB


## Part 4: Test ImageBind with Simple Example

In [4]:
# Test with text-only first
text_list = [
    "A dramatic moment",
    "Crowd cheering loudly", 
    "A funny scene",
    "An emotional speech"
]

inputs = {
    ModalityType.TEXT: data.load_and_transform_text(text_list, device)
}

with torch.no_grad():
    embeddings = model(inputs)

text_embeddings = embeddings[ModalityType.TEXT]

print(f"Text embeddings shape: {text_embeddings.shape}")
print(f"Embedding dimension: {text_embeddings.shape[1]}")

# Compute similarity between texts
similarity = torch.softmax(text_embeddings @ text_embeddings.T, dim=-1)
print("\nText similarity matrix:")
print(similarity)

print("\n✓ ImageBind is working!")

Text embeddings shape: torch.Size([4, 1024])
Embedding dimension: 1024

Text similarity matrix:
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]], device='cuda:0')

✓ ImageBind is working!


## Part 5: Video Segmentation Module

In [5]:
from decord import VideoReader, cpu
import numpy as np

def segment_video(video_path, window_size=2, stride=1):
    """
    Segment video into overlapping windows.
    
    Args:
        video_path: Path to video file
        window_size: Segment duration in seconds
        stride: Step between segments in seconds
    
    Returns:
        List of (start_time, end_time) tuples
    """
    vr = VideoReader(video_path, ctx=cpu(0))
    fps = vr.get_avg_fps()
    duration = len(vr) / fps
    
    segments = []
    t = 0
    while t + window_size <= duration:
        segments.append((t, t + window_size))
        t += stride
    
    print(f"Video duration: {duration:.1f}s")
    print(f"Segments created: {len(segments)}")
    print(f"Window size: {window_size}s, Stride: {stride}s")
    
    return segments, fps, vr

print("✓ Segmentation function ready")

✓ Segmentation function ready


## Part 6: Frame Extraction for Vision

In [6]:
def extract_frames(vr, start_time, end_time, fps, num_frames=8):
    """
    Extract frames from video segment.
    
    Args:
        vr: VideoReader object
        start_time: Start time in seconds
        end_time: End time in seconds  
        fps: Video FPS
        num_frames: Number of frames to sample
    
    Returns:
        Numpy array of frames
    """
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    
    # Sample frames uniformly
    frame_indices = np.linspace(start_frame, end_frame - 1, num_frames, dtype=int)
    
    # Extract frames
    frames = vr.get_batch(frame_indices).asnumpy()
    
    return frames

print("✓ Frame extraction function ready")

✓ Frame extraction function ready


## Part 7: Audio Extraction

In [7]:
import librosa
import soundfile as sf
import subprocess
import os

def extract_audio_segment(video_path, start_time, end_time, sr=16000):
    """
    Extract audio segment from video.
    
    Args:
        video_path: Path to video
        start_time: Start time in seconds
        end_time: End time in seconds
        sr: Sample rate
    
    Returns:
        Audio waveform as numpy array
    """
    # Extract full audio first (cache this for efficiency)
    audio_file = video_path.replace('.mp4', '.wav')
    
    if not os.path.exists(audio_file):
        # Extract audio using ffmpeg
        cmd = [
            'ffmpeg', '-i', video_path,
            '-vn', '-acodec', 'pcm_s16le',
            '-ar', str(sr), '-ac', '1',
            audio_file, '-y'
        ]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # Load segment
    audio, _ = librosa.load(
        audio_file,
        sr=sr,
        offset=start_time,
        duration=end_time - start_time
    )
    
    return audio

print("✓ Audio extraction function ready")

✓ Audio extraction function ready


## Part 7B: TRANSCRIPT Extraction

In [12]:
def extract_transcript_with_timestamps(video_path, whisper_model):
    """
    Extract transcript with word-level timestamps using Whisper.
    
    Returns:
        List of segments with text and timestamps
    """
    print("Extracting transcript with Whisper...")
    
    # Transcribe with word timestamps
    result = whisper_model.transcribe(
        video_path,
        word_timestamps=True,
        verbose=False
    )
    
    # Extract segments
    segments = []
    for segment in result['segments']:
        segments.append({
            'start': segment['start'],
            'end': segment['end'],
            'text': segment['text'].strip()
        })
    
    print(f"✓ Extracted {len(segments)} text segments")
    
    return segments, result['text']

print("✓ Transcript extraction function ready")

✓ Transcript extraction function ready


## Align Text to Video Segments

In [13]:
def align_text_to_segments(transcript_segments, video_segments):
    """
    Align transcript text to video segment timestamps.
    
    Args:
        transcript_segments: List from Whisper with text + timestamps
        video_segments: List of (path, start, end) tuples
    
    Returns:
        List of text snippets aligned to each video segment
    """
    aligned_texts = []
    
    for vid_path, vid_start, vid_end in video_segments:
        # Find all transcript segments that overlap with this video segment
        overlapping_text = []
        
        for trans_seg in transcript_segments:
            # Check if transcript segment overlaps with video segment
            if not (trans_seg['end'] < vid_start or trans_seg['start'] > vid_end):
                overlapping_text.append(trans_seg['text'])
        
        # Combine text from overlapping segments
        segment_text = " ".join(overlapping_text).strip()
        
        # If no text, use empty string
        if not segment_text:
            segment_text = ""
        
        aligned_texts.append(segment_text)
    
    print(f"✓ Aligned text to {len(aligned_texts)} video segments")
    
    return aligned_texts

## Segmentation

In [14]:
def segment_video_ffmpeg(video_path, output_dir="segments", window_size=2, stride=1):
    """Segment video into clips using FFmpeg."""
    # Get duration
    cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
           '-of', 'default=noprint_wrappers=1:nokey=1', video_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    duration = float(result.stdout.strip())
    
    os.makedirs(output_dir, exist_ok=True)
    
    segments = []
    t = 0
    idx = 0
    
    while t + window_size <= duration:
        seg_path = os.path.join(output_dir, f"segment_{idx:04d}.mp4")
        
        cmd = ['ffmpeg', '-y', '-ss', str(t), '-i', video_path,
               '-t', str(window_size), '-c', 'copy', seg_path]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        segments.append((seg_path, t, t + window_size))
        t += stride
        idx += 1
    
    print(f"✓ Created {len(segments)} video segments")
    return segments

In [15]:
import torch
import whisper
import numpy as np
import os
from imagebind import data
from imagebind.models.imagebind_model import ModalityType
import subprocess
import json
from datetime import datetime
import torch.nn.functional as F

print("✓ All imports ready")



# Load Whisper Model

# Load Whisper for transcription
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")  # Options: tiny, base, small, medium, large
print("✓ Whisper loaded (base model)")


✓ All imports ready
Loading Whisper model...
✓ Whisper loaded (base model)


## Part 8: Complete Segment Processing Pipeline

In [ ]:
import tempfile
from PIL import Image

def process_segment_trimodal(vr, video_path, start_time, end_time, fps, segment_text, num_frames=8):
    """
    Process a single video segment through ImageBind WITH TEXT.
    
    Returns:
        Dictionary with vision, audio, text, and combined embeddings
    """
    # Extract frames
    frames = extract_frames(vr, start_time, end_time, fps, num_frames)
    
    # Save frames temporarily for ImageBind
    frame_paths = []
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, frame in enumerate(frames):
            frame_path = os.path.join(tmpdir, f"frame_{i}.jpg")
            Image.fromarray(frame).save(frame_path)
            frame_paths.append(frame_path)
        
        # Extract audio
        audio = extract_audio_segment(video_path, start_time, end_time)
        
        # Save audio temporarily
        audio_path = os.path.join(tmpdir, "audio.wav")
        sf.write(audio_path, audio, 16000)
        
        # Prepare inputs for ImageBind (NOW WITH TEXT!)
        inputs = {
            ModalityType.VISION: data.load_and_transform_vision_data(
                frame_paths[:1],  # ImageBind uses single image per sample #we should view how to deal with this!!!!
                device
            ),
            ModalityType.AUDIO: data.load_and_transform_audio_data(
                [audio_path],
                device
            )
        }
        
        # Add text if available (some segments ynajem ykounesh fihom text yaani famesh klem b tharoura)
        if segment_text:
            inputs[ModalityType.TEXT] = data.load_and_transform_text(
                [segment_text],  # Must be a list
                device
            )
        
        # Get embeddings
        with torch.no_grad():
            embeddings = model(inputs)
        
        # Extract embeddings
        vision_emb = embeddings[ModalityType.VISION].cpu().numpy()
        audio_emb = embeddings[ModalityType.AUDIO].cpu().numpy()
        
        # Text embedding (might not exist for all segments)
        if ModalityType.TEXT in embeddings:
            text_emb = embeddings[ModalityType.TEXT].cpu().numpy()
        else:
            text_emb = np.zeros_like(vision_emb)  # Zero embedding if no text
        
        # Combined embedding (mean of all three)
        combined_emb = (vision_emb + audio_emb + text_emb) / 3
    
    return {
        'vision': vision_emb,      # 1024-d feature vector
        'audio': audio_emb,        # 1024-d feature vector
        'text': text_emb,          # 1024-d feature vector
        'combined': combined_emb,  # averaged (optional)
        'start': start_time,
        'end': end_time
    }

print("✓ Trimodal segment processing pipeline ready")

✓ Trimodal segment processing pipeline ready


## Part 10: Full Video Processing Pipeline

In [25]:
from tqdm import tqdm

def extract_trimodal_features(video_path, window_size=2, stride=1, num_frames=8):
    """
    Extract ONLY features (embeddings) from video.
    No scoring - just raw Vision + Audio + Text features.
    
    Returns:
        (results, full_transcript)
    """
    print(f"Processing: {video_path}\n")
    
    # Step 1: Extract transcript
    print("Step 1: Extracting transcript with Whisper...")
    transcript_segments, full_text = extract_transcript_with_timestamps(
        video_path, whisper_model
    )
    
    # Step 2: Segment video
    print("\nStep 2: Segmenting video...")
    segments, fps, vr = segment_video(video_path, window_size, stride)
    
    # Step 3: Align text to segments
    segment_tuples = [(None, start, end) for start, end in segments]
    print("\nStep 3: Aligning text to segments...")
    segment_texts = align_text_to_segments(transcript_segments, segment_tuples)
    text_count = sum(1 for t in segment_texts if t)
    print(f"✓ {text_count}/{len(segment_texts)} segments have text")
    
    # Step 4: Extract features (NO SCORING)
    print("\nStep 4: Extracting trimodal features...")
    results = []
    
    for idx, (start, end) in enumerate(tqdm(segments, desc="Extracting features")):
        try:
            # Get embeddings only
            seg_data = process_segment_trimodal(
                vr, video_path, start, end, fps, 
                segment_texts[idx], num_frames
            )
            
            # Store ONLY features (no scores)
            results.append({
                'start': start,
                'end': end,
                'text': segment_texts[idx],
                'vision_emb': seg_data['vision'],    # Raw 1024-d vector
                'audio_emb': seg_data['audio'],      # Raw 1024-d vector
                'text_emb': seg_data['text'],        # Raw 1024-d vector
                'combined_emb': seg_data['combined'] # Optional averaged
            })
            
        except Exception as e:
            print(f"\nError at {start}-{end}s: {e}")
            continue
    
    print(f"\n✓ Extracted features for {len(results)} segments")
    print(f"  Each segment has: vision (1024d) + audio (1024d) + text (1024d)")
    
    return results, full_text

print("✓ Feature extraction pipeline ready (no scoring)")

✓ Feature extraction pipeline ready (no scoring)


## Part 11: Test with Your Video

In [26]:
#test with the feature extraction fct only
# UPDATE THIS PATH
video_path = "/content/drive/MyDrive/PFA/data/Obama_Yes_we_can.mp4"

if os.path.exists(video_path):
    print("Starting TRIMODAL feature extraction...\n")
    
    # Extract features only (no scoring)
    results, full_transcript = extract_trimodal_features(
        video_path,
        window_size=2,
        stride=1,
        num_frames=8
    )
    
    print(f"\n✓ Extracted features for {len(results)} segments")
    print(f"✓ Transcript: {len(full_transcript)} characters")
    print(f"\n✓ Ready for custom scoring!")
    
else:
    print(f"Video not found: {video_path}")

Starting TRIMODAL feature extraction...

Processing: /content/drive/MyDrive/PFA/data/Obama_Yes_we_can.mp4

Step 1: Extracting transcript with Whisper...
Extracting transcript with Whisper...
Detected language: English


100%|██████████| 8939/8939 [00:02<00:00, 3210.10frames/s]


✓ Extracted 20 text segments

Step 2: Segmenting video...
Video duration: 89.3s
Segments created: 88
Window size: 2s, Stride: 1s

Step 3: Aligning text to segments...
✓ Aligned text to 88 video segments
✓ 86/88 segments have text

Step 4: Extracting trimodal features...


Extracting features: 100%|██████████| 88/88 [03:11<00:00,  2.17s/it]



✓ Extracted features for 88 segments
  Each segment has: vision (1024d) + audio (1024d) + text (1024d)

✓ Extracted features for 88 segments
✓ Transcript: 944 characters

✓ Ready for custom scoring!


## Part 13: Save Results

In [ ]:
#saving embeddings as they are
import json
from datetime import datetime
import numpy as np
import os

if 'results' in locals():
    results_dir = "/content/drive/MyDrive/PFA/ImageBind/results"
    os.makedirs(results_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Save metadata + text (NO SCORES)
    metadata_file = os.path.join(results_dir, f"features_trimodal_{timestamp}.json")
    
    output_data = {
        'video': video_path,
        'timestamp': datetime.now().isoformat(),
        'model': 'ImageBind-huge + Whisper-base',
        'modalities': ['vision', 'audio', 'text'],
        'embedding_dim': 1024,
        'config': {
            'window_size': 2,
            'stride': 1,
            'num_frames': 8
        },
        'full_transcript': full_transcript,
        'num_segments': len(results),
        'segments': [
            {
                'start': r['start'],
                'end': r['end'],
                'text': r['text']
            }
            for r in results
        ]
    }
    
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Metadata saved to: {metadata_file}")
    
    # Save ALL embeddings
    embeddings_file = os.path.join(results_dir, f"features_trimodal_{timestamp}_embeddings.npz")
    
    np.savez_compressed(
        embeddings_file,
        vision=np.array([r['vision_emb'] for r in results]),      # (N, 1024)
        audio=np.array([r['audio_emb'] for r in results]),        # (N, 1024)
        text=np.array([r['text_emb'] for r in results]),          # (N, 1024)
        times=np.array([(r['start'], r['end']) for r in results]) # (N, 2)
    )
    
    print(f"✓ Embeddings saved to: {embeddings_file}")
    
    # Save readable summary
    summary_file = os.path.join(results_dir, f"features_trimodal_{timestamp}_summary.txt")
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("TRIMODAL FEATURE EXTRACTION\n")
        f.write("Vision + Audio + Text Embeddings\n")
        f.write("=" * 70 + "\n")
        f.write(f"Video: {video_path}\n")
        f.write(f"Model: ImageBind\n")
        f.write(f"Embedding dimension: 1024\n")
        f.write(f"Number of segments: {len(results)}\n")
        f.write("=" * 70 + "\n\n")
        
        f.write("FULL TRANSCRIPT:\n")
        f.write("-" * 70 + "\n")
        f.write(full_transcript)
        f.write("\n\n")
        
        f.write("SEGMENTS WITH TEXT:\n")
        f.write("=" * 70 + "\n")
        for r in results:
            if r['text']:
                f.write(f"[{r['start']:6.1f}s - {r['end']:6.1f}s]: \"{r['text']}\"\n")
    
    print(f"✓ Summary saved to: {summary_file}")
    print(f"\n✓ All files saved in: {results_dir}")
    print("\n You can now load embeddings and apply your own scoring!")

✓ Metadata saved to: /content/drive/MyDrive/PFA/ImageBind/results/features_trimodal_20260214_102147.json
✓ Embeddings saved to: /content/drive/MyDrive/PFA/ImageBind/results/features_trimodal_20260214_102147_embeddings.npz
✓ Summary saved to: /content/drive/MyDrive/PFA/ImageBind/results/features_trimodal_20260214_102147_summary.txt

✓ All files saved in: /content/drive/MyDrive/PFA/ImageBind/results

 You can now load embeddings and apply your own scoring!


---
## Summary

### What This Notebook Does:

1.  **Loads ImageBind** - Unified embedding space for vision + audio + text
2.  **Segments video** - 2-second windows with 1-second stride
3.  **Extracts multimodal embeddings** - Vision and audio per segment
4.  **Computes highlight scores** - Semantic similarity to highlight anchors
5.  **Visualizes results** - Timeline of scores
6.  **Saves outputs** - JSON scores + embeddings for later use

### Key Advantages of ImageBind:

- **Unified space**: Vision, audio, text aligned automatically
- **Audio-aware**: Critical for detecting cheers, reactions, music
- **Zero-shot**: No training needed for highlight detection
- **Efficient**: ~1.2GB model, manageable on GPU
- **Clean**: Single embedding per segment, easy to score

### Next Steps:

1. **Temporal smoothing**: Apply Gaussian filter to scores
2. **Peak detection**: Find local maxima
3. **Continuous segment extraction**: Extract 30-60 sec highlights
4. **Optional text integration**: Add Whisper transcription
5. **Scoring refinement**: Tune weights (vision vs audio)

### Architecture Recap:

```
Video → Segments (2s each)
         ↓
    ImageBind Encoder
    ├── Vision: 8 frames → Embedding
    └── Audio: 2s waveform → Embedding
    └── Text: 2s transcript → Embedding

         ↓
    Peak Detection → Highlights
```

**You now have a working ImageBind pipeline for highlight extraction!**



## 1. **Current Approach Strengths**

**a) Unified Trimodal Embeddings**

* You are using **ImageBind** to embed vision, audio, and text in a **shared latent space**.
* This aligns with AudioCLIP ([10]) and ImageBind-LLM ([13]) principles: cross-modal zero-shot capabilities.
* You correctly extract:

  * Vision: 8 sampled frames per segment
  * Audio: waveform per segment
  * Text: aligned Whisper transcription
* **Benefit:** You can compute semantic similarity across modalities to rank segment relevance.

**b) Flexible Video Segmentation**

* Segments are created with **window size** (2s) and **stride** (1s), which allows overlapping context for highlights.
* You also have **FFmpeg-based clipping**, giving you both embeddings and actual video clips for export.
* **Benefit:** Fine-grained segmentation supports accurate highlight detection.

**c) Modular Pipeline**

* Segmentation, frame extraction, audio extraction, text alignment, and embedding are separate steps.
* You store embeddings, transcript, and segment metadata, making scoring and peak detection later easier.

**d) Lightweight Compared to Video-LLaMA2**

* Your model (~1.2GB) is lighter than full video-language LLMs like Video-LLaMA ([14]), which require multi-GPU setups and huge RAM.
* Zero-shot scoring is **fast**, unlike training-intensive Video-LLaMA or ImageBind-LLM fine-tuning.

---

## 2. **Current Limitations / Observations**

**a) Vision Embedding Sampling**

* Currently only **1 image per segment** (`frame_paths[:1]`) is passed to ImageBind for vision embedding.
* ImageBind can encode multiple images per batch; averaging embeddings for more frames could improve temporal context representation.

**b) Text Embedding Sparsity**

* Some segments may have **no text**. You set text embedding to zeros in that case.
* This works, but segments with only audio or visual cues may be underrepresented in combined embeddings.

**c) Audio Feature Representation**

* You extract raw waveform embeddings. Audio peaks like laughter or applause are captured indirectly.
* AudioCLIP ([10]) suggests using **spectrogram-based audio encoders** can improve sound-event detection, which could enhance highlight scoring.

**d) No Scoring Yet**

* Currently, you only extract embeddings. You haven’t implemented:

  * Cross-modal scoring against "highlight anchors"
  * Temporal smoothing
  * Peak detection
* Without scoring, segments can’t be automatically ranked for virality or engagement.

**e) Temporal Modeling**

* ImageBind embeddings are **per-segment averages**, ignoring fine-grained temporal dynamics.
* Video-LLaMA ([14]) and ImageBind-LLM ([13]) inject temporal or cross-frame modeling (Q-Former or bind networks). Your current method doesn’t consider sequential modeling, which could miss context.

**f) Multi-modality Alignment**

* Using a simple mean of vision, audio, text embeddings is straightforward but not adaptive.
* Weighting modalities based on content (e.g., emphasize audio for applause) is not yet implemented.

**g) Scaling Considerations**

* ImageBind is light and fast for single GPU, but segmenting long videos at 1s stride could still produce **hundreds of segments**, which may become memory intensive.
* Unlike OpenCLIP/ImageBind scaling studies ([11,12]), you haven’t explored batch processing or memory-optimized embedding.

---
## 3. **Immediate Improvements**

1. **Vision Embedding**

   * Pass **all sampled frames** per segment, not just the first.
   * Average the embeddings or apply a lightweight temporal aggregation (mean or attention).

2. **Audio Embedding**

   * Consider converting waveform → log-mel spectrogram to match AudioCLIP input style.
   * This could improve highlight scoring on auditory cues.

3. **Text Embedding**

   * If no text, consider **propagating neighboring segments’ text** or weighting the embedding differently.

4. **Adaptive Fusion**

   * Weighted combination of vision, audio, text embeddings.
   * Could use simple learned weights or use cosine similarity against a “highlight anchor vector”.

5. **Highlight Scoring**

   * Compute **similarity to highlight templates**, or define scores for energy/laughter/text sentiment.
   * Apply **temporal smoothing** and **peak detection**.

6. **Memory and Speed**

   * Process segments in batches, especially for long videos.
   * Save intermediate embeddings to **disk** to prevent recomputation.


